# Task B3 — Pose Project: Yoga Pose Classifier (Teachable Machine → Python)



In [2]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
import time, json, csv, threading
from pathlib import Path
import numpy as np, cv2
import tensorflow as tf
try:
    import tf_keras as keras      # Teachable Machine .h5 files need legacy Keras 2 -> pip install tf_keras
except ImportError as e:
    raise ImportError("Run:  pip install tf_keras   (same minor version as TensorFlow), then restart the kernel") from e
np.set_printoptions(suppress=True)
print("TensorFlow", tf.__version__, "| tf_keras", keras.__version__, "| OpenCV", cv2.__version__)

HERE = Path.cwd(); MODELS = HERE / "models"; OUT = HERE / "output"; OUT.mkdir(exist_ok=True)
assert (MODELS / "model.json").exists(), "Run this notebook from the pose/ folder"

TensorFlow 2.21.0 | tf_keras 2.21.0 | OpenCV 5.0.0


## B3.3 · Load the exported model and PoseNet
`load_tm_pose_head` rebuilds the TF.js `Dense(100, relu) → Dropout(0.5) → Dense(5, softmax, no bias)` classifier in Keras and copies the trained weights from `weights.bin`.

In [3]:
import base64
from PIL import Image

KEYPOINTS = ["nose","leftEye","rightEye","leftEar","rightEar","leftShoulder","rightShoulder","leftElbow","rightElbow",
             "leftWrist","rightWrist","leftHip","rightHip","leftKnee","rightKnee","leftAnkle","rightAnkle"]
SKELETON = [(0,1),(0,2),(1,3),(2,4),(5,6),(5,7),(7,9),(6,8),(8,10),(5,11),(6,12),(11,12),(11,13),(13,15),(12,14),(14,16)]
INPUT_RES, STRIDE = 257, 16
USE_SIGMOID = True

# 1. Teachable Machine pose head (TF.js layers model) -> Keras
def load_tm_pose_head(model_dir):
    spec = json.load(open(Path(model_dir) / "model.json"))
    buf = (Path(model_dir) / spec["weightsManifest"][0]["paths"][0]).read_bytes()
    w, off = {}, 0
    for e in spec["weightsManifest"][0]["weights"]:
        n = int(np.prod(e["shape"])); w[e["name"]] = np.frombuffer(buf, "<f4", n, off).reshape(e["shape"]); off += 4 * n
    layers = spec["modelTopology"]["config"]["layers"]
    in_dim = layers[0]["config"]["batch_input_shape"][1]
    head = keras.Sequential([
        keras.layers.Input((in_dim,)),
        keras.layers.Dense(layers[0]["config"]["units"], activation="relu", name="dense_1"),
        keras.layers.Dropout(layers[1]["config"]["rate"]),
        keras.layers.Dense(layers[2]["config"]["units"], activation="softmax", use_bias=False, name="dense_2"),
    ], name="tm_pose_head")
    head.get_layer("dense_1").set_weights([w["dense_Dense3/kernel"], w["dense_Dense3/bias"]])
    head.get_layer("dense_2").set_weights([w["dense_Dense4/kernel"]])
    return head

# 2. PoseNet MobileNetV1 (TF.js graph model) -> plain TensorFlow function
def load_tfjs_posenet(model_json):
    """Minimal interpreter for the PoseNet TF.js graph model (only fused Conv2D / DepthwiseConv2D + bias + ReLU6)."""
    d = Path(model_json).parent
    spec = json.load(open(model_json))
    buf = b"".join((d / p).read_bytes() for p in spec["weightsManifest"][0]["paths"])
    consts, off = {}, 0
    for e in spec["weightsManifest"][0]["weights"]:
        n = int(np.prod(e["shape"])); consts[e["name"]] = tf.constant(np.frombuffer(buf, "<f4", n, off).reshape(e["shape"])); off += 4 * n
    nodes = spec["modelTopology"]["node"]
    b64 = lambda s: base64.b64decode(s).decode()
    @tf.autograph.experimental.do_not_convert
    def run(image):
        vals = {"image": image}
        for n in nodes:
            if n["op"] == "Placeholder": continue
            if n["op"] == "Const": vals[n["name"]] = consts[n["name"]]; continue
            a = n["attr"]; x, k, b = (vals[i.split(":")[0]] for i in n["input"][:3])
            st = [int(v) for v in a["strides"]["list"]["i"]][1:3]; dil = [int(v) for v in a["dilations"]["list"]["i"]][1:3]
            pad = b64(a["padding"]["s"])
            if n["op"] == "_FusedConv2D": y = tf.nn.conv2d(x, k, strides=st, padding=pad, dilations=dil)
            else: y = tf.nn.depthwise_conv2d(x, k, strides=[1] + st + [1], padding=pad, dilations=dil)
            fused = [b64(s) for s in a["fused_ops"]["list"]["s"]]
            if "BiasAdd" in fused: y = tf.nn.bias_add(y, b)
            if "Relu6" in fused: y = tf.nn.relu6(y)
            vals[n["name"]] = y
        return vals["MobilenetV1/heatmap_2/BiasAdd"], vals["MobilenetV1/offset_2/BiasAdd"]
    return tf.function(run, input_signature=[tf.TensorSpec([1, INPUT_RES, INPUT_RES, 3], tf.float32)])

# ---- 3. Frame -> 14,739 features + keypoints ------------------------------------------------------------------
def extract_pose(posenet, rgb):
    """rgb: HxWx3 uint8. Returns (features[1,14739], keypoints[17 x (x, y, score)] in frame pixel coordinates)."""
    h, w = rgb.shape[:2]; s = max(h, w); ph, pw = (s - h) // 2, (s - w) // 2
    sq = np.zeros((s, s, 3), np.uint8); sq[ph:ph + h, pw:pw + w] = rgb                       # pad to square (as TM does)
    x = tf.image.resize(sq[None].astype("float32"), (INPUT_RES, INPUT_RES)) / 127.5 - 1.0     # MobileNet: [-1, 1]
    heat, off = posenet(x)
    heat = tf.sigmoid(heat) if USE_SIGMOID else heat
    feats = tf.reshape(tf.concat([heat, off], axis=3), (1, -1))                               # 17x17x(17+34) -> 14,739
    hm, of = heat[0].numpy(), off[0].numpy(); kps = []
    for k in range(17):
        r, c = np.unravel_index(hm[:, :, k].argmax(), hm.shape[:2])
        y = r * STRIDE + of[r, c, k]; xx = c * STRIDE + of[r, c, k + 17]
        kps.append((xx / INPUT_RES * s - pw, y / INPUT_RES * s - ph, float(hm[r, c, k])))
    return feats, kps

def draw_skeleton(img, kps, min_score=0.5, color=(0, 255, 255)):
    for a, b in SKELETON:
        if kps[a][2] > min_score and kps[b][2] > min_score:
            cv2.line(img, (int(kps[a][0]), int(kps[a][1])), (int(kps[b][0]), int(kps[b][1])), color, 2, cv2.LINE_AA)
    for x, y, sc in kps:
        if sc > min_score: cv2.circle(img, (int(x), int(y)), 4, (0, 0, 255), -1, cv2.LINE_AA)
    return img

In [4]:
meta = json.load(open(MODELS / "metadata.json")); labels = meta["labels"]; print("PoseNet settings:", meta["modelSettings"]["posenet"]); print("Labels:", labels)
head = load_tm_pose_head(MODELS)
posenet = load_tfjs_posenet(MODELS / "posenet" / "model.json")
assert head.output_shape[-1] == len(labels)

PoseNet settings: {'architecture': 'MobileNetV1', 'outputStride': 16, 'inputResolution': 257, 'multiplier': 0.75}
Labels: ['Downdog', 'Goddness', 'Plank', 'Tree', 'Warrior2']



## B4.4 · Model introspection (pose model)

In [5]:
head.summary()
n_params = lambda ws: sum(int(np.prod(w.shape)) for w in ws)
print("input_shape :", head.input_shape, " output_shape:", head.output_shape)
print("layers      :", len(head.layers), "| params:", f"{head.count_params():,}", "| trainable:", f"{n_params(head.trainable_weights):,}")
print("PoseNet feature extractor (frozen, NOT trained by you): 17 heatmaps + 34 offsets on a 17x17 grid =", 17 * 17 * 51, "features")

Model: "tm_pose_head"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_1 (Dense)             (None, 100)               1474000   
                                                                 
 dropout (Dropout)           (None, 100)               0         
                                                                 
 dense_2 (Dense)             (None, 5)                 500       
                                                                 
Total params: 1474500 (5.62 MB)
Trainable params: 1474500 (5.62 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
input_shape : (None, 14739)  output_shape: (None, 5)
layers      : 3 | params: 1,474,500 | trainable: 1,474,500
PoseNet feature extractor (frozen, NOT trained by you): 17 heatmaps + 34 offsets on a 17x17 grid = 14739 features


In [9]:
def classify_rgb(rgb):
    feats, kps = extract_pose(posenet, rgb)
    probs = head(feats, training=False).numpy()[0]
    return probs, kps


### Validate on the Kaggle dataset (recommended)
Point `DATASET_DIR` at the folder containing one sub-folder per pose (`Downdog`, `Goddess`/`Goddness`, `Plank`, `Tree`, `Warrior2`; matching is case-insensitive and tolerant of the *Goddess* spelling). If accuracy is far below your Teachable Machine preview accuracy, try `USE_SIGMOID = False` in the library cell above.

In [10]:
DATASET_DIR = None   # e.g. r"C:/data/yoga_poses/TEST"
if DATASET_DIR:
    norm = lambda s: s.lower().replace("goddess", "goddness")
    lab = {norm(l): i for i, l in enumerate(labels)}; ok = n = 0
    for cls in sorted(Path(DATASET_DIR).iterdir()):
        if norm(cls.name) not in lab: continue
        for f in list(cls.glob("*.*"))[:30]:
            im = cv2.imread(str(f))
            if im is None: continue
            n += 1; ok += int(classify_rgb(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))[0].argmax()) == lab[norm(cls.name)]
    print(f"Accuracy on {n} images: {ok / max(n, 1):.2%}")

## B3.4 / B3.5 · Webcam deployment + Yoga Coach

```
APPLICATION: Yoga Coach
 - What it does : runs PoseNet on every frame, classifies the pose, draws the skeleton and shows
                  the *target* pose name, the detected pose and confidence.  When detected == target and
                  confidence > 80 % the frame border turns GREEN and a hold timer counts up.
 - How to use   : stand ~2 m from the camera so your whole body is visible.
                  n = next target pose   q = quit      (output saved to output/b3_pose_result.mp4)
 - Optional     : put reference photos in reference/<label>.jpg (e.g. reference/Tree.jpg) to show them in the corner.
 - Poses        : Downdog, Goddness, Plank, Tree, Warrior2
```
Run with `coach=False` for the plain classifier (B3.4).

In [11]:
def load_references(size=140):
    refs = {}
    for l in labels:
        p = HERE / "reference" / f"{l}.jpg"
        if p.exists(): refs[l] = cv2.resize(cv2.imread(str(p)), (size, size))
    return refs
REFS = load_references()

def pose_loop(camera=0, mirror=True, coach=True, min_conf=0.8, max_seconds=300, save_video=OUT / "b3_pose_result.mp4"):
    cap = cv2.VideoCapture(camera)
    if not cap.isOpened(): raise RuntimeError("Cannot open webcam")
    ok, frame = cap.read(); H, W = frame.shape[:2]
    vw = cv2.VideoWriter(str(save_video), cv2.VideoWriter_fourcc(*"mp4v"), 20, (W, H)) if save_video else None
    target, hold_start, t0 = 0, None, time.time()
    try:
        while time.time() - t0 < max_seconds:
            ok, frame = cap.read()
            if not ok: break
            if mirror: frame = cv2.flip(frame, 1)
            probs, kps = classify_rgb(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            i = int(probs.argmax()); conf = float(probs[i]); out = draw_skeleton(frame.copy(), kps)
            cv2.rectangle(out, (0, 0), (W, 34), (0, 0, 0), -1)
            cv2.putText(out, f"Detected: {labels[i]} {conf:.0%}", (8, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2, cv2.LINE_AA)
            if coach:
                matched = i == target and conf > min_conf
                hold_start = (hold_start or time.time()) if matched else None
                cv2.putText(out, f"Target: {labels[target]}  (n = next)", (W // 2 - 20, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2, cv2.LINE_AA)
                if matched:
                    cv2.rectangle(out, (0, 0), (W - 1, H - 1), (0, 255, 0), 10)
                    cv2.putText(out, f"HOLD {time.time() - hold_start:.1f}s", (W // 2 - 70, H - 20), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 3, cv2.LINE_AA)
                if labels[target] in REFS:
                    r = REFS[labels[target]]; out[40:40 + r.shape[0], W - r.shape[1] - 6:W - 6] = r
            if vw: vw.write(out)
            cv2.imshow("B3 Pose Classifier  (q = quit, n = next pose)", out)
            k = cv2.waitKey(1) & 0xFF
            if k == ord("q"): break
            if k == ord("n"): target = (target + 1) % len(labels); hold_start = None
    finally:
        cap.release(); cv2.destroyAllWindows(); cv2.waitKey(1)
        if vw: vw.release(); print("saved", save_video)

In [ ]:
RUN_WEBCAM = True
if RUN_WEBCAM: pose_loop(coach=True)